##Student Data Retrieval LLM

In [ ]:
!pip install -U langchain langchain-google-genai

In [3]:
import os
from google.colab import userdata

# Google AI Studio
key = userdata.get("GEMINI_API_KEY")
os.environ["GOOGLE_API_KEY"] = key

In [4]:
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
import sqlite3

In [5]:
conn = sqlite3.connect("students.db")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    student_id TEXT PRIMARY KEY,
    name TEXT,
    department TEXT,
    python INTEGER,
    database INTEGER,
    ai INTEGER,
    web INTEGER
)
""")

students = [
    ("22CS045", "Dhanushya", "Computer Science", 85, 72, 90, 78),
    ("22CS046", "Rahul", "Computer Science", 65, 70, 68, 72),
    ("22CS047", "Priya", "Information Technology", 92, 88, 95, 90),
    ("22CS048", "Arun", "Information Technology", 55, 60, 58, 62),
    ("22CS049", "Meena", "Computer Science", 78, 85, 80, 88)
]

cursor.executemany("""
INSERT OR IGNORE INTO students
(student_id, name, department, python, database, ai, web)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", students)

conn.commit()
conn.close()

In [6]:
@tool
def get_student_info(student_id: str) -> str:
    """
    Get the name and department of a student using their student ID.
    Use this tool when the user asks for a student's name or department.
    """

    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()

    cursor.execute("""
    SELECT name, department
    FROM students
    WHERE student_id = ?
    """, (student_id,))

    student = cursor.fetchone()

    conn.close()

    if student is None:
        return "Student not found."

    name, department = student

    return f"Name: {name}, Department: {department}"

In [7]:
@tool
def get_student_marks(student_id: str) -> str:
    """
    Get the marks of a student.
    Returns Python, Database, AI, and Web marks.
    Use this tool when the user asks for a student's marks.
    """

    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()

    cursor.execute("""
    SELECT python, database, ai, web
    FROM students
    WHERE student_id = ?
    """, (student_id,))

    marks = cursor.fetchone()

    conn.close()

    if marks is None:
        return "Student not found."

    python_mark, database_mark, ai_mark, web_mark = marks

    return (
        f"Python: {python_mark}, "
        f"Database: {database_mark}, "
        f"AI: {ai_mark}, "
        f"Web: {web_mark}"
    )

In [8]:
@tool
def calculator(expression: str) -> str:
    """
    Calculate a mathematical expression.
    Use this tool when the user needs total marks or average marks.
    """

    try:
        result = eval(expression)
        return str(result)

    except Exception as e:
        return f"Calculation error: {e}"

In [9]:
@tool
def get_passing_rules() -> str:
    """
    Get the university passing rules.
    Use this tool when the user asks whether a student satisfies
    the university passing requirements.
    """

    return """
    University Passing Rules:
    - Minimum overall average: 40%
    - Minimum mark in each subject: 35%
    """

In [10]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

agent = create_agent(
    model=llm,
    tools=[
        get_student_info,
        get_student_marks,
        calculator,
        get_passing_rules
    ],
    system_prompt="""
    You are a student information assistant.
    Use the available tools to answer student-related questions.
    Do not invent student information.
    When a question requires information from the database, use the appropriate tool.
    When a calculation is required, use the calculator tool.
    When checking passing eligibility, use the student's marks, the university passing rules, and the calculator when necessary.
    Decide yourself which tools are required and in what order.
    Do not assume information that is not provided by the tools.
    """
)

In [ ]:
question = input("Ask a question: ")

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": question
    }]
})

answer = result["messages"][-1].content

if isinstance(answer, list):
    answer = "\n".join(
        item["text"]
        for item in answer
        if isinstance(item, dict) and item.get("type") == "text"
    )

print(answer)